# 08 · Stability, Subgroup Audit and Findings

Bring model, capacity and source limitations together without presenting synthetic results as production evidence.

## Reading guide

This notebook is part of a connected workflow. It states the decision being made, shows the supporting checks and records limitations alongside the result. Source files are never modified in place.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("FIFAR_DATA_DIR", PROJECT_ROOT / "data" / "raw" / "FiFAR"))
REPORTS = PROJECT_ROOT / "reports"
IMAGES = PROJECT_ROOT / "images"

sns.set_theme(style="whitegrid")
CORAL = "#F08FA0"
TEAL = "#0E6268"
DARK = "#15262B"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Set FIFAR_DATA_DIR to the extracted official FiFAR directory before running this notebook."
    )

In [ ]:
base = pd.read_csv(DATA_ROOT / 'alert_data' / 'Base.csv').dropna().copy()
metrics = json.loads((REPORTS / 'model_metrics.json').read_text())
review = json.loads((REPORTS / 'review_strategy_metrics.json').read_text())

## 1. Temporal stability

In [ ]:
monthly = base[~base.month.eq(4)].groupby("month")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean")
monthly.assign(fraud_rate_percent=lambda frame: (frame["fraud_rate"] * 100).round(2))

The observed fraud rate changes across the supplied months. This supports temporal monitoring and makes random-only validation inappropriate.

## 2. Audit fields

In [ ]:
audit_fields = ["income", "customer_age", "employment_status", "housing_status"]
pd.DataFrame({
    "field": audit_fields,
    "used_by_primary_model": [False] * len(audit_fields),
    "retained_for_subgroup_audit": [True] * len(audit_fields),
})

Excluding these fields does not guarantee fairness. Remaining features may act as proxies, and the synthetic dataset cannot establish legal or regulatory suitability.

In [ ]:
age_profile = base.groupby("customer_age")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean")
age_profile.assign(fraud_rate_percent=lambda frame: (frame["fraud_rate"] * 100).round(2))

In [ ]:
employment_profile = base.groupby("employment_status")["fraud_bool"].agg(applications="size", fraud_cases="sum", fraud_rate="mean")
employment_profile.assign(fraud_rate_percent=lambda frame: (frame["fraud_rate"] * 100).round(2))

These are descriptive target distributions, not model fairness metrics. Full subgroup evaluation requires saved test predictions, minimum-group support rules and uncertainty intervals.

## 3. Consolidated final-month result

In [ ]:
selected = metrics["models"]["hist_gradient_boosting"]
three_percent = next(row for row in selected["test_capacity"] if row["review_share"] == .03)
pd.Series({
    "test_applications": metrics["split"]["test_rows"],
    "average_precision": selected["test"]["average_precision"],
    "roc_auc": selected["test"]["roc_auc"],
    **three_percent,
})

## 4. Alert-review result

In [ ]:
pd.DataFrame(review['strategy_summary']).sort_values('mean_accuracy', ascending=False)

## 5. Findings

- The supplied application target is rare, so accuracy is not informative for the ranking model.
- Temporal evaluation exposes changing fraud prevalence.
- Gradient boosting improves ranking and fraud recovery over logistic regression.
- At three percent capacity, the queue captures 533 of 1,428 fraud cases.
- That queue still contains 2,372 legitimate applications and misses 895 fraud cases.
- Across 25 test teams, risk-band assignment reduces mean false positives by about 180 compared with random capacity allocation, while recall falls by about 1.7 percentage points.
- Analyst variation makes assignment and workload part of the detection problem.

## 6. Limitations

- All applications and analysts are synthetic.
- The supplied base CSV is truncated during month 4.
- The current model comparison is deliberately limited.
- No monetary loss or investigation cost is supplied.
- Protected-field exclusion does not prove fairness.
- Analyst skill is estimated from a short historical window and may change.
- Results do not demonstrate production or regulatory readiness.

## Recommendation

Use the model as a ranking component, not an automatic rejection rule. Monitor queue precision, fraud recovery, monthly drift, subgroup behaviour and analyst-assignment outcomes. Select the assignment objective only after defining the relative cost of false positives and missed fraud.